In [0]:
df = spark.read.table("marathos.bronze.races")
df.count()

In [0]:
df.select("Event distance/length", "Athlete performance").distinct().show(200, truncate=False)

In [0]:
from pyspark.sql.functions import col

df_clean = df.filter(~col("Event distance/length").like("%d"))

df_clean.filter(col("Event distance/length").like("%d")).count()

In [0]:
df_clean = df.filter(
    ~(col("Event distance/length").like("%km") & col("Athlete performance").like("%km")) &
    ~(col("Event distance/length").like("%mi") & col("Athlete performance").like("%mi")) &
    ~(col("Event distance/length").like("%h") & col("Athlete performance").like("%h"))
)


In [0]:
df.printSchema()

In [0]:
df_clean = df.filter(~col("Athlete performance").like("%d%"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import when

df_clean = (
    df_clean
    .withColumn("Athlete performance", F.regexp_replace("Athlete performance", "h", ""))
    .withColumn("performance_split", F.split(F.col("Athlete performance"), ":"))
    .withColumn(
        "Athlete performance",
        when(
            col("Event distance/length").like("%h"),
            F.regexp_replace(col("Athlete performance"), " km", "").cast("double")
        ).otherwise(
            F.col("performance_split")[0].cast("double")
            + F.col("performance_split")[1].cast("double") / 60
            + F.col("performance_split")[2].cast("double") / 3600
        )
    )
    .drop("performance_split")
)




In [0]:
df_clean.select("Event distance/length", "Athlete performance").show(20, truncate=False)

In [0]:
df_clean.filter(col("Event distance/length").like("%h")).select("Event distance/length", "Athlete performance").show(10, truncate=False)